# AFHQ Autoencoder — Analysis

Training now happens in `train.py`, a **three-phase curriculum** for the DC-AE-style `PetPaletteAE`:

    uv run python projects/afhq/autoencoder/train.py

1. **Phase 1** — L1 + LPIPS @ 64×64, train the *full* model.
2. **Phase 2** — L1 + LPIPS @ 256×256, train *only* the latent bottleneck (`to_latent`, `from_latent`).
3. **Phase 3** — L1 + LPIPS + PatchGAN @ 64×64, train *only* the output head (`out_from_channels`).

The three phases share one in-memory model, so weights carry over, and each phase writes a stable checkpoint under `/mnt/ai/runs/afhq/autoencoder/checkpoints/`. This notebook only loads the final `ae_phase3.ckpt` for analysis: reconstruction quality and metrics at 256×256.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ['retina', 'svg']
%load_ext autoreload
%autoreload 2

import os

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torchmetrics.functional.image import (
    peak_signal_noise_ratio,
    structural_similarity_index_measure,
)
from torchvision.transforms.functional import to_pil_image
from torchvision.utils import make_grid

from chimera.data import AFHQDataModule
from chimera.models import PetPaletteAE

# Keep the LPIPS-VGG and FID-Inception weights off the small root disk.
os.environ.setdefault("HF_HOME", "/mnt/ai/data/hf")
os.environ.setdefault("TORCH_HOME", "/mnt/ai/data/torch")

RUN_DIR = "/mnt/ai/runs/afhq/autoencoder"
DATA_DIR = "/mnt/ai/data"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_float32_matmul_precision("high")

## Data

The 256×256 validation view (the resolution the curriculum always validates at). `normalize=False` keeps pixels in `[0, 1]` to match the decoder's sigmoid output and the `data_range=1.0` metrics (LPIPS/FID take `[0, 1]` inputs too). The default `seed` lines the split up with training.

In [ ]:
dm = AFHQDataModule(
    data_dir=DATA_DIR,
    image_size=256,
    batch_size=32,
    num_workers=8,
    normalize=False,
)
dm.prepare_data()
dm.setup("fit")
val_loader = dm.val_dataloader()  # validation is always 256x256
print(f"classes={dm.classes}")

In [ ]:
images, _ = next(iter(val_loader))

grid = make_grid(images[:8].clamp(0, 1), nrow=8)
plt.figure(figsize=(12, 2))
plt.imshow(to_pil_image(grid))
plt.title("Sample AFHQ Images (256x256)")
plt.axis("off")
plt.show()

## Load checkpoint

Load a fresh `PetPaletteAE` (same config as `train.py`) and restore the final phase-3 weights from the training run.

In [ ]:
ckpt = torch.load(
    f"{RUN_DIR}/checkpoints/ae_phase3.ckpt", map_location="cpu", weights_only=False
)
# The checkpoint is a LightningModule state_dict; keep only the AE (drop the
# discriminator / LPIPS / FID sub-metrics) by taking the "model." prefix.
model_state = {
    k.removeprefix("model."): v
    for k, v in ckpt["state_dict"].items()
    if k.startswith("model.")
}

model = PetPaletteAE(
    in_channels=3, latent_dim=4, base_channels=32, fsq_levels=[8, 5, 5, 5]
)
model.load_state_dict(model_state)
# Lightning leaves the model on CPU after fit/test; move it back for inference.
model.to(DEVICE).float().eval()
print(f"loaded AE ({sum(p.numel() for p in model.parameters()):,} params, epoch {ckpt['epoch']})")

## Analysis

Compare 256×256 validation images against their reconstructions from the fully-trained model.

In [ ]:
@torch.no_grad()
def recon_metrics(model, loader, device):
    maes, psnrs, ssims = [], [], []
    for x, _ in loader:
        x = x.to(device)
        recon = model(x).float()
        maes.append(F.l1_loss(recon, x).item())
        psnrs.append(peak_signal_noise_ratio(recon, x, data_range=1.0).item())
        ssims.append(structural_similarity_index_measure(recon, x, data_range=1.0).item())
    return sum(maes) / len(maes), sum(psnrs) / len(psnrs), sum(ssims) / len(ssims)


mae, psnr, ssim = recon_metrics(model, val_loader, DEVICE)
print(f"val MAE {mae:.4f}, PSNR {psnr:.2f} dB, SSIM {ssim:.3f}")

images, _ = next(iter(val_loader))
with torch.no_grad():
    recons = model(images.to(DEVICE)).float().cpu()
    diffs = (images - recons).abs()

n = 8
# Row 0: originals, row 1: reconstructions, row 2: differences.
batch = torch.cat([images[:n], recons[:n], diffs[:n]]).clamp(0, 1)
grid = make_grid(batch, nrow=n)
plt.figure(figsize=(20, 6))
plt.imshow(to_pil_image(grid))
plt.title("Original (top) / Reconstruction (middle) / Diff (bottom) @ 256x256")
plt.axis("off")
plt.show()